## Section 0 - Virtual Environment Setup and Package Installation

This notebook creates a dedicated local environment for the 2-class YOLO26 ONNX workflow, installs the required packages into that environment, and registers it as a Jupyter kernel. Run these setup cells first, then switch the notebook kernel before continuing.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = Path.cwd().resolve()
VENV_DIR = ROOT / "yolo26_env"
if os.name == "nt":
    venv_python = VENV_DIR / "Scripts" / "python.exe"
    venv_pip = VENV_DIR / "Scripts" / "pip.exe"
else:
    venv_python = VENV_DIR / "bin" / "python"
    venv_pip = VENV_DIR / "bin" / "pip"

if VENV_DIR.exists():
    print(f"Virtual environment already exists at {VENV_DIR}; skipping creation.")
else:
    subprocess.run([sys.executable, "-m", "venv", str(VENV_DIR)], check=True)
    print(f"Created virtual environment at {VENV_DIR}")

assert venv_python.exists(), f"Expected venv Python executable at {venv_python}"
assert venv_pip.exists(), f"Expected venv pip executable at {venv_pip}"
print(f"Venv python executable: {venv_python}")
print(f"Venv pip executable: {venv_pip}")
print("Section 0A completed: the virtual environment is ready.")

Virtual environment already exists at C:\Users\mepix\coding\detectx\yolo26_env; skipping creation.
Venv python executable: C:\Users\mepix\coding\detectx\yolo26_env\Scripts\python.exe
Venv pip executable: C:\Users\mepix\coding\detectx\yolo26_env\Scripts\pip.exe
Section 0A completed: the virtual environment is ready.


In [2]:
from pathlib import Path
import os
import subprocess

ROOT = Path.cwd().resolve()
VENV_DIR = ROOT / "yolo26_env"
if os.name == "nt":
    venv_python = VENV_DIR / "Scripts" / "python.exe"
    venv_pip = VENV_DIR / "Scripts" / "pip.exe"
else:
    venv_python = VENV_DIR / "bin" / "python"
    venv_pip = VENV_DIR / "bin" / "pip"

pytorch_packages = ["torch", "torchvision"]
other_packages = [
    "ultralytics==8.4.2",
    "onnx>=1.12.0,<2.0.0",
    "onnxruntime",
    "onnxslim>=0.1.71",
    "numpy<2.0",
    "opencv-python",
    "pyyaml",
    "ipykernel",
]

pytorch_install_cmd = [str(venv_pip), "install", "--no-cache-dir", "--upgrade"]
if os.name == "nt":
    pytorch_install_cmd.extend(["--index-url", "https://download.pytorch.org/whl/cu128"])
pytorch_install_cmd.extend(pytorch_packages)

subprocess.run(pytorch_install_cmd, check=True)
subprocess.run([str(venv_pip), "install", "--no-cache-dir", "--upgrade", *other_packages], check=True)
subprocess.run(
    [
        str(venv_python),
        "-m",
        "ipykernel",
        "install",
        "--user",
        "--name",
        "yolo26_env",
        "--display-name",
        "Python (yolo26_env)",
    ],
    check=True,
)

if os.name == "nt":
    print("Windows note: this notebook stays on the native ONNX deployment path and does not attempt a TFLite conversion.")
print("Section 0B completed: packages are installed and the kernel is registered.")

Windows note: this notebook stays on the native ONNX deployment path and does not attempt a TFLite conversion.
Section 0B completed: packages are installed and the kernel is registered.


> **Important:** Go to **Kernel -> Change Kernel** and select **`Python (yolo26_env)`**, then restart the kernel before running anything below this point.
>
> If you skip this step, the remaining cells will run against your system Python instead of the dedicated YOLO26 environment.

## Section 1 - Environment Verification

This section verifies that the notebook is running inside the dedicated environment, checks whether PyTorch can see a CUDA-capable GPU for training, and confirms that ONNX Runtime is available for later ONNX validation and Raspberry Pi deployment.

In [3]:
import os
import platform

import onnxruntime as ort
import torch
from ultralytics import __version__ as ultralytics_version

print(f"Python version: {platform.python_version()}")
print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch CUDA runtime: {torch.version.cuda}")
print(f"Ultralytics version: {ultralytics_version}")
print(f"ONNX Runtime version: {ort.__version__}")

cuda_available = torch.cuda.is_available()
print(f"PyTorch CUDA available: {cuda_available}")
if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_properties = torch.cuda.get_device_properties(0)
    vram_gb = gpu_properties.total_memory / (1024 ** 3)
    print(f"PyTorch GPU name: {gpu_name}")
    print(f"Approximate VRAM: {vram_gb:.2f} GB")
else:
    print("PyTorch GPU name: none detected")
    print("Approximate VRAM: 0.00 GB")
    print("WARNING: CUDA is not available. Training will fall back to CPU.")

print(f"Available ONNX Runtime providers: {ort.get_available_providers()}")
print(f"ULTRALYTICS_SKIP_REQUIREMENTS_CHECKS: {os.environ.get('ULTRALYTICS_SKIP_REQUIREMENTS_CHECKS', '(not set)')}")

DEVICE = "cuda:0" if cuda_available else "cpu"
print(f"Selected device: {DEVICE}")
print("Section 1 completed: the runtime environment was verified.")

Python version: 3.12.3
PyTorch version: 2.11.0+cu128
PyTorch CUDA runtime: 12.8
Ultralytics version: 8.4.2
ONNX Runtime version: 1.26.0
PyTorch CUDA available: True
PyTorch GPU name: NVIDIA GeForce RTX 5070
Approximate VRAM: 11.94 GB
Available ONNX Runtime providers: ['AzureExecutionProvider', 'CPUExecutionProvider']
ULTRALYTICS_SKIP_REQUIREMENTS_CHECKS: (not set)
Selected device: cuda:0
Section 1 completed: the runtime environment was verified.


## Section 2 - Path Configuration

This section defines the full 2-class workflow configuration, including the source dataset, filtered dataset, structured train-val-test dataset, ONNX export folder, and the class mapping that keeps only `undefected` and `dirt_defect`.

In [4]:
from pathlib import Path
from datetime import datetime
import json

ROOT = Path.cwd().resolve()
DATASET_VARIANT = "rgb"  # change to "greyscale" to train on the greyscale retrain_v3 dataset
RETRAIN_VERSION = "v4"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
SOURCE_DATASET_DIR = ROOT / f"raw_dataset_balanced_retrain_{RETRAIN_VERSION}_{DATASET_VARIANT}"
FILTERED_DATASET_DIR = ROOT / f"raw_dataset_balanced_retrain_{RETRAIN_VERSION}_{DATASET_VARIANT}_two_class"
STRUCTURED_DATASET_DIR = ROOT / f"structured_dataset_two_class_retrain_{RETRAIN_VERSION}_{DATASET_VARIANT}"
RUNS_DIR = ROOT / "runs"
RUN_NAME = f"yolo26n_defects_2class_end2end_retrain_{RETRAIN_VERSION}_{DATASET_VARIANT}_{RUN_TIMESTAMP}"
EXPORT_DIR = ROOT / "exported_models" / f"yolo26_onnx_end2end_two_class_retrain_{RETRAIN_VERSION}_{DATASET_VARIANT}_{RUN_TIMESTAMP}"
ONNX_FILENAME = f"{RUN_NAME}.onnx"
ONNX_PATH = EXPORT_DIR / ONNX_FILENAME
DATA_YAML_PATH = STRUCTURED_DATASET_DIR / "data.yaml"
DATASET_MANIFEST_PATH = STRUCTURED_DATASET_DIR / "dataset_manifest.json"
PREVIEW_PATH = EXPORT_DIR / f"preview_{RUN_TIMESTAMP}.jpg"

CLASS_NAMES = {
    0: "undefected",
    1: "dirt_defect",
}
SOURCE_CLASS_IDS = {0, 1, 2}
DROPPED_CLASS_IDS = {2}
KEPT_CLASS_IDS = tuple(sorted(CLASS_NAMES))
RANDOM_SEED = 42
TRAIN_IMGSZ = 960
EXPORT_IMGSZ = 640
FORCE_REBUILD_DATASET = False
SPLIT_RATIOS = {"train": 0.80, "val": 0.10, "test": 0.10}
IMAGE_EXTENSIONS = {
    ".bmp",
    ".gif",
    ".jpeg",
    ".jpg",
    ".png",
    ".tif",
    ".tiff",
    ".webp",
}
LABEL_EXT = ".txt"

DATASET_CONFIG_SIGNATURE = {
    "source_dataset_dir": str(SOURCE_DATASET_DIR.resolve()),
    "filtered_dataset_dir": str(FILTERED_DATASET_DIR.resolve()),
    "structured_dataset_dir": str(STRUCTURED_DATASET_DIR.resolve()),
    "kept_class_ids": list(KEPT_CLASS_IDS),
    "dropped_class_ids": sorted(DROPPED_CLASS_IDS),
    "class_names": {str(class_id): class_name for class_id, class_name in CLASS_NAMES.items()},
    "random_seed": RANDOM_SEED,
    "split_ratios": SPLIT_RATIOS,
}

for output_dir in [RUNS_DIR, EXPORT_DIR]:
    output_dir.mkdir(parents=True, exist_ok=True)

assert SOURCE_DATASET_DIR.exists(), f"Source dataset directory was not found: {SOURCE_DATASET_DIR}"

print(f"ROOT: {ROOT}")
print(f"DATASET_VARIANT: {DATASET_VARIANT}")
print(f"RETRAIN_VERSION: {RETRAIN_VERSION}")
print(f"RUN_TIMESTAMP: {RUN_TIMESTAMP}")
print(f"SOURCE_DATASET_DIR: {SOURCE_DATASET_DIR}")
print(f"FILTERED_DATASET_DIR: {FILTERED_DATASET_DIR}")
print(f"STRUCTURED_DATASET_DIR: {STRUCTURED_DATASET_DIR}")
print(f"RUNS_DIR: {RUNS_DIR}")
print(f"RUN_NAME: {RUN_NAME}")
print(f"EXPORT_DIR: {EXPORT_DIR}")
print(f"ONNX_PATH: {ONNX_PATH}")
print(f"DATA_YAML_PATH: {DATA_YAML_PATH}")
print(f"DATASET_MANIFEST_PATH: {DATASET_MANIFEST_PATH}")
print(f"PREVIEW_PATH: {PREVIEW_PATH}")
print("Active 2-class config:")
print(json.dumps(DATASET_CONFIG_SIGNATURE, indent=2))
print("Section 2 completed: project paths and constants are configured.")


ROOT: C:\Users\mepix\coding\detectx
DATASET_VARIANT: rgb
RETRAIN_VERSION: v4
RUN_TIMESTAMP: 20260610_150601
SOURCE_DATASET_DIR: C:\Users\mepix\coding\detectx\raw_dataset_balanced_retrain_v4_rgb
FILTERED_DATASET_DIR: C:\Users\mepix\coding\detectx\raw_dataset_balanced_retrain_v4_rgb_two_class
STRUCTURED_DATASET_DIR: C:\Users\mepix\coding\detectx\structured_dataset_two_class_retrain_v4_rgb
RUNS_DIR: C:\Users\mepix\coding\detectx\runs
RUN_NAME: yolo26n_defects_2class_end2end_retrain_v4_rgb_20260610_150601
EXPORT_DIR: C:\Users\mepix\coding\detectx\exported_models\yolo26_onnx_end2end_two_class_retrain_v4_rgb_20260610_150601
ONNX_PATH: C:\Users\mepix\coding\detectx\exported_models\yolo26_onnx_end2end_two_class_retrain_v4_rgb_20260610_150601\yolo26n_defects_2class_end2end_retrain_v4_rgb_20260610_150601.onnx
DATA_YAML_PATH: C:\Users\mepix\coding\detectx\structured_dataset_two_class_retrain_v4_rgb\data.yaml
DATASET_MANIFEST_PATH: C:\Users\mepix\coding\detectx\structured_dataset_two_class_retrain

## Section 3 - Audit, Filter, Stratify, and Create `data.yaml`

This section audits every YOLO label in the balanced source dataset, enforces the single-object assumption used by this workflow, drops class `2` (`shape_defect`), rebuilds the filtered and structured datasets whenever the manifest no longer matches the active config, and writes the new 2-class `data.yaml` file.

In [5]:
from __future__ import annotations

import json
import random
import shutil
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import yaml


@dataclass(frozen=True)
class Sample:
    image_path: Path
    label_path: Path
    class_id: int
    rows: list[tuple[int, float, float, float, float]]


def parse_label_rows(label_path: Path) -> list[tuple[int, float, float, float, float]]:
    text = label_path.read_text(encoding="utf-8").strip()
    if not text:
        raise ValueError(f"Label file is empty: {label_path}")

    rows: list[tuple[int, float, float, float, float]] = []
    for line_number, line in enumerate(text.splitlines(), start=1):
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(
                f"Invalid YOLO row in {label_path} on line {line_number}: expected 5 fields, got {len(parts)}"
            )

        class_id_text, x_text, y_text, width_text, height_text = parts
        try:
            class_id = int(class_id_text)
        except ValueError as exc:
            raise ValueError(
                f"Invalid class id in {label_path} on line {line_number}: {class_id_text}"
            ) from exc

        if class_id not in SOURCE_CLASS_IDS:
            raise ValueError(
                f"Unexpected class id in {label_path} on line {line_number}: {class_id}. "
                f"Expected one of {sorted(SOURCE_CLASS_IDS)}"
            )

        coordinates: list[float] = []
        for field_name, field_value in (
            ("x_center", x_text),
            ("y_center", y_text),
            ("width", width_text),
            ("height", height_text),
        ):
            try:
                numeric_value = float(field_value)
            except ValueError as exc:
                raise ValueError(
                    f"Invalid {field_name} value in {label_path} on line {line_number}: {field_value}"
                ) from exc
            if not 0.0 <= numeric_value <= 1.0:
                raise ValueError(
                    f"Out-of-range {field_name} value in {label_path} on line {line_number}: {numeric_value}"
                )
            coordinates.append(numeric_value)

        rows.append((class_id, *coordinates))

    return rows


def format_rows(rows: list[tuple[int, float, float, float, float]]) -> str:
    return "\n".join(
        f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
        for class_id, x_center, y_center, width, height in rows
    ) + "\n"


def build_stem_index(paths: list[Path], kind: str) -> dict[str, Path]:
    index: dict[str, Path] = {}
    for path in paths:
        key = path.stem.casefold()
        if key in index:
            raise ValueError(f"Multiple {kind} files share the same stem: {index[key].name}, {path.name}")
        index[key] = path
    return index


def load_existing_manifest() -> dict | None:
    if not DATASET_MANIFEST_PATH.exists():
        return None
    try:
        return json.loads(DATASET_MANIFEST_PATH.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return None


def scan_label_counts(label_paths: list[Path]) -> tuple[int, Counter[int]]:
    row_counts: Counter[int] = Counter()
    file_count = 0
    for label_path in label_paths:
        file_count += 1
        for class_id, *_ in parse_label_rows(label_path):
            row_counts[class_id] += 1
    return file_count, row_counts


def remove_dir_if_exists(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)


def collect_source_samples() -> tuple[list[Sample], Counter[int], int]:
    image_paths = sorted(
        path for path in SOURCE_DATASET_DIR.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not image_paths:
        raise FileNotFoundError(f"No supported image files were found in {SOURCE_DATASET_DIR}")

    label_paths = sorted(path for path in SOURCE_DATASET_DIR.iterdir() if path.is_file() and path.suffix.lower() == LABEL_EXT)
    image_by_stem = build_stem_index(image_paths, "image")
    label_by_stem = build_stem_index(label_paths, "label")

    missing_labels = [image_path for image_path in image_paths if image_path.stem.casefold() not in label_by_stem]
    orphan_labels = [label_path for label_path in label_paths if label_path.stem.casefold() not in image_by_stem]
    assert not missing_labels, f"Found {len(missing_labels)} image(s) without matching labels in {SOURCE_DATASET_DIR}"
    assert not orphan_labels, f"Found {len(orphan_labels)} label file(s) without matching images in {SOURCE_DATASET_DIR}"

    source_class_counts: Counter[int] = Counter()
    filtered_samples: list[Sample] = []
    dropped_sample_count = 0

    for image_path in image_paths:
        label_path = label_by_stem[image_path.stem.casefold()]
        rows = parse_label_rows(label_path)
        if len(rows) != 1:
            raise AssertionError(
                f"Expected exactly one object per label for stratified splitting, but found {len(rows)} rows in {label_path}"
            )

        class_ids = {class_id for class_id, *_ in rows}
        if len(class_ids) != 1:
            raise AssertionError(
                f"Expected one class id per label file, but found multiple classes in {label_path}: {sorted(class_ids)}"
            )

        source_class_id = rows[0][0]
        source_class_counts[source_class_id] += 1

        if source_class_id in DROPPED_CLASS_IDS:
            dropped_sample_count += 1
            continue

        filtered_samples.append(
            Sample(
                image_path=image_path,
                label_path=label_path,
                class_id=source_class_id,
                rows=rows,
            )
        )

    return filtered_samples, source_class_counts, dropped_sample_count


def stratified_split(samples: list[Sample]) -> tuple[dict[str, list[Sample]], dict[str, Counter[int]]]:
    samples_by_class: defaultdict[int, list[Sample]] = defaultdict(list)
    for sample in samples:
        samples_by_class[sample.class_id].append(sample)

    split_samples = {"train": [], "val": [], "test": []}
    split_class_counts = {split_name: Counter() for split_name in split_samples}

    for class_id in sorted(samples_by_class):
        class_samples = sorted(samples_by_class[class_id], key=lambda sample: sample.image_path.name.casefold())
        class_rng = random.Random(RANDOM_SEED + class_id)
        class_rng.shuffle(class_samples)

        class_count = len(class_samples)
        train_count = int(class_count * SPLIT_RATIOS["train"])
        val_count = int(class_count * SPLIT_RATIOS["val"])
        test_count = class_count - train_count - val_count
        if min(train_count, val_count, test_count) <= 0:
            raise AssertionError(
                f"Class {class_id} does not have enough samples for a non-empty stratified split: {class_count}"
            )

        class_split_map = {
            "train": class_samples[:train_count],
            "val": class_samples[train_count:train_count + val_count],
            "test": class_samples[train_count + val_count:],
        }

        for split_name, split_subset in class_split_map.items():
            split_samples[split_name].extend(split_subset)
            split_class_counts[split_name][class_id] += len(split_subset)

    for split_name in split_samples:
        split_samples[split_name] = sorted(split_samples[split_name], key=lambda sample: sample.image_path.name.casefold())

    return split_samples, split_class_counts


def write_label_file(path: Path, rows: list[tuple[int, float, float, float, float]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(format_rows(rows), encoding="utf-8")


def rebuild_dataset(filtered_samples: list[Sample], source_class_counts: Counter[int], dropped_sample_count: int) -> dict:
    remove_dir_if_exists(FILTERED_DATASET_DIR)
    remove_dir_if_exists(STRUCTURED_DATASET_DIR)

    FILTERED_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    split_destinations = {}
    for split_name in ("train", "val", "test"):
        split_destinations[split_name] = (
            STRUCTURED_DATASET_DIR / "images" / split_name,
            STRUCTURED_DATASET_DIR / "labels" / split_name,
        )
        split_destinations[split_name][0].mkdir(parents=True, exist_ok=True)
        split_destinations[split_name][1].mkdir(parents=True, exist_ok=True)

    filtered_class_counts: Counter[int] = Counter(sample.class_id for sample in filtered_samples)
    split_samples, split_class_counts = stratified_split(filtered_samples)

    for sample in filtered_samples:
        shutil.copy2(sample.image_path, FILTERED_DATASET_DIR / sample.image_path.name)
        write_label_file(FILTERED_DATASET_DIR / sample.label_path.name, sample.rows)

    for split_name, samples_in_split in split_samples.items():
        image_dest_dir, label_dest_dir = split_destinations[split_name]
        for sample in samples_in_split:
            shutil.copy2(sample.image_path, image_dest_dir / sample.image_path.name)
            write_label_file(label_dest_dir / sample.label_path.name, sample.rows)

    data_config = {
        "path": str(STRUCTURED_DATASET_DIR),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "nc": len(CLASS_NAMES),
        "names": CLASS_NAMES,
    }
    DATA_YAML_PATH.parent.mkdir(parents=True, exist_ok=True)
    with DATA_YAML_PATH.open("w", encoding="utf-8") as yaml_file:
        yaml.safe_dump(data_config, yaml_file, sort_keys=False, allow_unicode=False)

    manifest = {
        "config_signature": DATASET_CONFIG_SIGNATURE,
        "source_pair_count": int(sum(source_class_counts.values())),
        "source_class_counts": {str(class_id): int(count) for class_id, count in sorted(source_class_counts.items())},
        "dropped_sample_count": int(dropped_sample_count),
        "filtered_pair_count": int(len(filtered_samples)),
        "filtered_class_counts": {str(class_id): int(count) for class_id, count in sorted(filtered_class_counts.items())},
        "split_counts": {split_name: int(len(samples_in_split)) for split_name, samples_in_split in split_samples.items()},
        "split_class_counts": {
            split_name: {str(class_id): int(count) for class_id, count in sorted(class_counts.items())}
            for split_name, class_counts in split_class_counts.items()
        },
    }
    DATASET_MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest


filtered_samples, source_class_counts, dropped_sample_count = collect_source_samples()
expected_filtered_class_counts = Counter(sample.class_id for sample in filtered_samples)
existing_manifest = load_existing_manifest()

rebuild_reasons: list[str] = []
if FORCE_REBUILD_DATASET:
    rebuild_reasons.append("FORCE_REBUILD_DATASET is True")
if existing_manifest is None:
    rebuild_reasons.append("dataset manifest is missing or unreadable")
elif existing_manifest.get("config_signature") != DATASET_CONFIG_SIGNATURE:
    rebuild_reasons.append("dataset manifest does not match the active 2-class config")

for required_path in [
    FILTERED_DATASET_DIR,
    STRUCTURED_DATASET_DIR / "images" / "train",
    STRUCTURED_DATASET_DIR / "images" / "val",
    STRUCTURED_DATASET_DIR / "images" / "test",
    STRUCTURED_DATASET_DIR / "labels" / "train",
    STRUCTURED_DATASET_DIR / "labels" / "val",
    STRUCTURED_DATASET_DIR / "labels" / "test",
    DATA_YAML_PATH,
]:
    if not required_path.exists():
        rebuild_reasons.append(f"required path is missing: {required_path}")

if rebuild_reasons:
    print("Rebuilding the 2-class dataset for the following reason(s):")
    for reason in rebuild_reasons:
        print(f"- {reason}")
    manifest = rebuild_dataset(filtered_samples, source_class_counts, dropped_sample_count)
else:
    manifest = existing_manifest
    print("Existing 2-class dataset matches the active config; skipping rebuild.")

filtered_label_paths = sorted(FILTERED_DATASET_DIR.glob(f"*{LABEL_EXT}"))
filtered_file_count, filtered_scan_counts = scan_label_counts(filtered_label_paths)
assert set(filtered_scan_counts).issubset(set(KEPT_CLASS_IDS)), (
    f"Filtered dataset contains unexpected class ids: {sorted(set(filtered_scan_counts) - set(KEPT_CLASS_IDS))}"
)
assert filtered_file_count == manifest["filtered_pair_count"], "Filtered dataset file count does not match the manifest."

structured_file_counts = {}
structured_class_counts = {}
for split_name in ("train", "val", "test"):
    split_label_paths = sorted((STRUCTURED_DATASET_DIR / "labels" / split_name).glob(f"*{LABEL_EXT}"))
    split_file_count, split_row_counts = scan_label_counts(split_label_paths)
    assert set(split_row_counts).issubset(set(KEPT_CLASS_IDS)), (
        f"Split {split_name} contains unexpected class ids: {sorted(set(split_row_counts) - set(KEPT_CLASS_IDS))}"
    )
    assert all(split_row_counts[class_id] > 0 for class_id in KEPT_CLASS_IDS), (
        f"Split {split_name} is missing one or more classes: {dict(split_row_counts)}"
    )
    structured_file_counts[split_name] = split_file_count
    structured_class_counts[split_name] = {class_id: int(split_row_counts[class_id]) for class_id in sorted(split_row_counts)}

assert sum(structured_file_counts.values()) == filtered_file_count, "Structured split counts do not sum to the filtered dataset size."

print(f"Source class counts: {dict(sorted(source_class_counts.items()))}")
print(f"Dropped class ids: {sorted(DROPPED_CLASS_IDS)}")
print(f"Dropped sample count: {dropped_sample_count}")
print(f"Filtered class counts: {dict(sorted(expected_filtered_class_counts.items()))}")
print(f"Structured split file counts: {structured_file_counts}")
print(f"Structured split class counts: {structured_class_counts}")
print("dataset_manifest.json contents:")
print(json.dumps(manifest, indent=2))
print("data.yaml contents:")
print(DATA_YAML_PATH.read_text(encoding="utf-8"))
print("Section 3 completed: the 2-class dataset, manifest, and data.yaml are ready.")

Existing 2-class dataset matches the active config; skipping rebuild.
Source class counts: {0: 2776, 1: 2776}
Dropped class ids: [2]
Dropped sample count: 0
Filtered class counts: {0: 2776, 1: 2776}
Structured split file counts: {'train': 4440, 'val': 554, 'test': 558}
Structured split class counts: {'train': {0: 2220, 1: 2220}, 'val': {0: 277, 1: 277}, 'test': {0: 279, 1: 279}}
dataset_manifest.json contents:
{
  "config_signature": {
    "source_dataset_dir": "C:\\Users\\mepix\\coding\\detectx\\raw_dataset_balanced_retrain_v4_rgb",
    "filtered_dataset_dir": "C:\\Users\\mepix\\coding\\detectx\\raw_dataset_balanced_retrain_v4_rgb_two_class",
    "structured_dataset_dir": "C:\\Users\\mepix\\coding\\detectx\\structured_dataset_two_class_retrain_v4_rgb",
    "kept_class_ids": [
      0,
      1
    ],
    "dropped_class_ids": [
      2
    ],
    "class_names": {
      "0": "undefected",
      "1": "dirt_defect"
    },
    "random_seed": 42,
    "split_ratios": {
      "train": 0.8,
   

## Section 4 - Train YOLO26n

This section trains a YOLO26 nano detector on the rebuilt 2-class dataset. The cell is safe to rerun because it reuses the expected `best.pt` weights file if training already completed.

In [6]:
from ultralytics import YOLO

BEST_PT_PATH = RUNS_DIR / RUN_NAME / "weights" / "best.pt"

if BEST_PT_PATH.exists():
    print(f"Existing trained weights found at {BEST_PT_PATH}; skipping training.")
else:
    model = YOLO("yolo26n.pt")
    train_results = model.train(
        data=str(DATA_YAML_PATH),
        epochs=300,
        imgsz=TRAIN_IMGSZ,
        batch=16,
        workers=4,
        device=DEVICE,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        patience=50,
        amp=DEVICE.startswith("cuda"),
        verbose=True,
        exist_ok=True,
    )
    if hasattr(train_results, "save_dir"):
        print(f"Training artifacts saved to: {train_results.save_dir}")

assert BEST_PT_PATH.exists(), f"Expected best weights file was not found: {BEST_PT_PATH}"
print(f"BEST_PT_PATH: {BEST_PT_PATH}")
print("Section 4 completed: training artifacts are available.")

New https://pypi.org/project/ultralytics/8.4.63 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.2  Python-3.12.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070, 12227MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\mepix\coding\detectx\structured_dataset_two_class_retrain_v4_rgb\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937,

## Section 5 - Evaluate the PyTorch Model on the Test Set

This section evaluates the best PyTorch checkpoint on the held-out 2-class test split at the training resolution so you can compare the native training-time metrics with the deployment-time ONNX metrics that follow.

In [7]:
from ultralytics import YOLO


def extract_headline_metrics(metrics) -> dict[str, float]:
    results_dict = getattr(metrics, "results_dict", {})
    box_metrics = getattr(metrics, "box", None)
    map50 = results_dict.get("metrics/mAP50(B)", getattr(box_metrics, "map50", float("nan")))
    map50_95 = results_dict.get("metrics/mAP50-95(B)", getattr(box_metrics, "map", float("nan")))
    precision = results_dict.get("metrics/precision(B)", getattr(box_metrics, "mp", float("nan")))
    recall = results_dict.get("metrics/recall(B)", getattr(box_metrics, "mr", float("nan")))
    return {
        "mAP50": float(map50),
        "mAP50-95": float(map50_95),
        "precision": float(precision),
        "recall": float(recall),
    }


pt_eval_model = YOLO(str(BEST_PT_PATH))
pt_metrics = pt_eval_model.val(
    data=str(DATA_YAML_PATH),
    split="test",
    imgsz=TRAIN_IMGSZ,
    device=DEVICE,
    verbose=True,
)

PT_TEST_METRICS = extract_headline_metrics(pt_metrics)
print("PyTorch test-set metrics at training resolution:")
for metric_name, metric_value in PT_TEST_METRICS.items():
    print(f"{metric_name}: {metric_value:.4f}")
print("Section 5 completed: PyTorch test-set evaluation finished.")

Ultralytics 8.4.2  Python-3.12.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070, 12227MiB)
YOLO26n summary (fused): 122 layers, 2,375,226 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 43.219.6 MB/s, size: 145.5 KB)
val: Scanning C:\Users\mepix\coding\detectx\structured_dataset_two_class_retrain_v4_rgb\labels\test... 558 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 558/558 2.2Kit/s 0.3s0.1ss
val: New cache created: C:\Users\mepix\coding\detectx\structured_dataset_two_class_retrain_v4_rgb\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 9.1it/s 3.9s<0.1s
                   all        558        558      0.976      0.965       0.99      0.967
            undefected        279        279      0.974      0.958       0.99      0.971
           dirt_defect        279        279      0.978      0.971       0.99      0.962
Speed: 2.3ms preprocess, 2.2ms inference, 

## Section 6 - Export to ONNX and Evaluate the Deployed Model

This section exports the best checkpoint to ONNX at `640`, explicitly pins the ONNX opset to `20`, skips Ultralytics requirement auto-installs to avoid the unwanted `onnxruntime-gpu` path on Windows, and evaluates the exported ONNX model on the full test split at the deployment resolution using CPU inference.

In [8]:
import os
import shutil
from pathlib import Path

from ultralytics import YOLO

os.environ["ULTRALYTICS_SKIP_REQUIREMENTS_CHECKS"] = "1"
print("ULTRALYTICS_SKIP_REQUIREMENTS_CHECKS=1")

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
BEST_PT_BACKUP_PATH = EXPORT_DIR / f"{RUN_NAME}_best.pt"

needs_export = not ONNX_PATH.exists() or ONNX_PATH.stat().st_mtime < BEST_PT_PATH.stat().st_mtime
if needs_export:
    export_model = YOLO(str(BEST_PT_PATH))
    export_result = export_model.export(
        format="onnx",
        imgsz=EXPORT_IMGSZ,
        opset=20,
        simplify=True,
    )
    export_result_path = Path(str(export_result))
    candidate_paths: list[Path] = []
    if export_result_path.exists() and export_result_path.is_file() and export_result_path.suffix == ".onnx":
        candidate_paths.append(export_result_path)
    search_roots = []
    if export_result_path.exists():
        search_roots.append(export_result_path if export_result_path.is_dir() else export_result_path.parent)
    search_roots.append(BEST_PT_PATH.parent)
    for search_root in search_roots:
        if search_root.exists():
            candidate_paths.extend(sorted(search_root.rglob("*.onnx")))
    candidate_paths = [path for path in candidate_paths if path.exists()]
    assert candidate_paths, f"No ONNX file was found after export. Raw export result: {export_result}"
    source_onnx_path = candidate_paths[0]
    if source_onnx_path.resolve() != ONNX_PATH.resolve():
        shutil.copy2(source_onnx_path, ONNX_PATH)
    else:
        ONNX_PATH = source_onnx_path
else:
    print(f"Existing ONNX export is up to date at {ONNX_PATH}; skipping export.")

assert ONNX_PATH.exists(), f"Expected ONNX file was not found: {ONNX_PATH}"
shutil.copy2(BEST_PT_PATH, BEST_PT_BACKUP_PATH)

onnx_size_mb = ONNX_PATH.stat().st_size / (1024 ** 2)
print(f"ONNX_PATH: {ONNX_PATH}")
print(f"ONNX file size: {onnx_size_mb:.2f} MB")
print(f"Backed up PyTorch weights to: {BEST_PT_BACKUP_PATH}")

onnx_eval_model = YOLO(str(ONNX_PATH))
onnx_metrics = onnx_eval_model.val(
    data=str(DATA_YAML_PATH),
    split="test",
    imgsz=EXPORT_IMGSZ,
    device="cpu",
    verbose=True,
)

ONNX_TEST_METRICS = extract_headline_metrics(onnx_metrics)
print("ONNX test-set metrics at deployment resolution:")
for metric_name, metric_value in ONNX_TEST_METRICS.items():
    print(f"{metric_name}: {metric_value:.4f}")
print("Section 6 completed: ONNX export and full-test ONNX evaluation finished.")


ULTRALYTICS_SKIP_REQUIREMENTS_CHECKS=1
Ultralytics 8.4.2  Python-3.12.3 torch-2.11.0+cu128 CPU (Intel Core i7-14700F)
YOLO26n summary (fused): 122 layers, 2,375,226 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from 'C:\Users\mepix\coding\detectx\runs\yolo26n_defects_2class_end2end_retrain_v4_rgb_20260610_150601\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.2 MB)
requirements: ULTRALYTICS_SKIP_REQUIREMENTS_CHECKS=1 detected, skipping requirements check.

ONNX: starting export with onnx 1.21.0 opset 20...


c:\Users\mepix\coding\detectx\yolo26_env\Lib\site-packages\torch\onnx\_internal\torchscript_exporter\symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.94...
ONNX: export success  1.6s, saved as 'C:\Users\mepix\coding\detectx\runs\yolo26n_defects_2class_end2end_retrain_v4_rgb_20260610_150601\weights\best.onnx' (9.4 MB)

Export complete (1.9s)
Results saved to C:\Users\mepix\coding\detectx\runs\yolo26n_defects_2class_end2end_retrain_v4_rgb_20260610_150601\weights
Predict:         yolo predict task=detect model=C:\Users\mepix\coding\detectx\runs\yolo26n_defects_2class_end2end_retrain_v4_rgb_20260610_150601\weights\best.onnx imgsz=640  
Validate:        yolo val task=detect model=C:\Users\mepix\coding\detectx\runs\yolo26n_defects_2class_end2end_retrain_v4_rgb_20260610_150601\weights\best.onnx imgsz=640 data=C:\Users\mepix\coding\detectx\structured_dataset_two_class_retrain_v4_rgb\data.yaml  
Visualize:       https://netron.app
ONNX_PATH: C:\Users\mepix\coding\detectx\exported_models\yolo26_onnx_end2end_two_class_retrain_v4_rgb_20260610_150601\yolo26n_defects_2class_end2end_retrain_v4_rgb_20260610_150601.o

## Section 7 - ONNX Runtime Sanity Inference and Preview Generation

This section loads the exported ONNX model with ONNX Runtime on CPU, runs a sanity-check inference on one test image, maps the predicted `xyxy` boxes from the letterboxed `640 x 640` space back to the original image coordinates, and saves an annotated preview image into the export folder.

In [9]:
import cv2
import numpy as np
import onnxruntime as ort

CONFIDENCE_THRESHOLD = 0.25


def letterbox_resize(image_bgr: np.ndarray, new_shape=(640, 640), color=(114, 114, 114)):
    original_height, original_width = image_bgr.shape[:2]
    scale = min(new_shape[0] / original_height, new_shape[1] / original_width)
    resized_width = int(round(original_width * scale))
    resized_height = int(round(original_height * scale))

    resized_image = cv2.resize(image_bgr, (resized_width, resized_height), interpolation=cv2.INTER_LINEAR)
    pad_width = new_shape[1] - resized_width
    pad_height = new_shape[0] - resized_height
    pad_left = int(round(pad_width / 2 - 0.1))
    pad_right = int(round(pad_width / 2 + 0.1))
    pad_top = int(round(pad_height / 2 - 0.1))
    pad_bottom = int(round(pad_height / 2 + 0.1))

    padded_image = cv2.copyMakeBorder(
        resized_image,
        pad_top,
        pad_bottom,
        pad_left,
        pad_right,
        cv2.BORDER_CONSTANT,
        value=color,
    )
    return padded_image, scale, (pad_left, pad_top)


def scale_boxes_back(boxes_xyxy: np.ndarray, original_shape: tuple[int, int], scale: float, padding: tuple[int, int]) -> np.ndarray:
    pad_left, pad_top = padding
    original_height, original_width = original_shape
    boxes = boxes_xyxy.astype(np.float32).copy()
    boxes[:, [0, 2]] -= pad_left
    boxes[:, [1, 3]] -= pad_top
    boxes /= scale
    boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, original_width - 1)
    boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, original_height - 1)
    return boxes


session = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
input_details = session.get_inputs()
output_details = session.get_outputs()

print(f"ONNX Runtime version for sanity check: {ort.__version__}")
print(f"Available ONNX Runtime providers: {ort.get_available_providers()}")
print(f"Session providers: {session.get_providers()}")
print("Input tensor details:")
for detail in input_details:
    print(f"name={detail.name}, shape={detail.shape}, type={detail.type}")
print("Output tensor details:")
for detail in output_details:
    print(f"name={detail.name}, shape={detail.shape}, type={detail.type}")

test_image_paths = sorted(
    path for path in (STRUCTURED_DATASET_DIR / "images" / "test").iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)
assert test_image_paths, "No test images were found for the ONNX sanity check."
sample_image_path = test_image_paths[0]
sample_bgr = cv2.imread(str(sample_image_path))
assert sample_bgr is not None, f"Failed to read test image: {sample_image_path}"

letterboxed_bgr, resize_scale, padding = letterbox_resize(
    sample_bgr,
    new_shape=(EXPORT_IMGSZ, EXPORT_IMGSZ),
    color=(114, 114, 114),
)
sample_rgb = cv2.cvtColor(letterboxed_bgr, cv2.COLOR_BGR2RGB)
input_tensor = sample_rgb.astype(np.float32) / 255.0
input_tensor = np.transpose(input_tensor, (2, 0, 1))
input_tensor = np.expand_dims(input_tensor, axis=0)

print(f"Sample test image: {sample_image_path.name}")
print(f"Letterbox resize scale: {resize_scale:.6f}, padding: {padding}")
print(f"Prepared ONNX tensor shape: {input_tensor.shape}")

raw_output = session.run(None, {input_details[0].name: input_tensor})[0]
print(f"Raw output tensor shape: {raw_output.shape}")
print("Expected native YOLO26 end-to-end layout for detection is approximately (1, 300, 6).")

detections = raw_output.astype(np.float32)
if detections.ndim == 3 and detections.shape[0] == 1:
    detections = detections[0]
if detections.ndim != 2:
    raise AssertionError(f"Unexpected detection output rank: {detections.ndim}. Raw shape: {raw_output.shape}")
if detections.shape[1] != 6 and detections.shape[0] == 6:
    detections = detections.T
if detections.shape[1] != 6:
    raise AssertionError(
        f"Unexpected detection output layout: {detections.shape}. "
        "Native YOLO26 end-to-end detection is expected to produce rows of length 6."
    )

scores = detections[:, 4]
keep_mask = scores >= CONFIDENCE_THRESHOLD
filtered_detections = detections[keep_mask].copy()
print(f"Final detections above confidence {CONFIDENCE_THRESHOLD}: {int(keep_mask.sum())}")
print("No manual NMS is applied here because the end-to-end export already produces final detections.")

annotated_bgr = sample_bgr.copy()
if filtered_detections.size:
    filtered_detections[:, :4] = scale_boxes_back(
        filtered_detections[:, :4],
        sample_bgr.shape[:2],
        resize_scale,
        padding,
    )
    top_indices = np.argsort(filtered_detections[:, 4])[::-1][:5]
    for rank, idx in enumerate(top_indices, start=1):
        x1, y1, x2, y2, score, class_id_value = filtered_detections[idx]
        class_id = int(np.clip(np.rint(class_id_value), 0, len(CLASS_NAMES) - 1))
        label = f"{CLASS_NAMES[class_id]} {float(score):.3f}"
        p1 = (int(round(x1)), int(round(y1)))
        p2 = (int(round(x2)), int(round(y2)))
        cv2.rectangle(annotated_bgr, p1, p2, (0, 255, 0), 2)
        cv2.putText(
            annotated_bgr,
            label,
            (p1[0], max(24, p1[1] - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2,
            cv2.LINE_AA,
        )
        print(
            f"Detection {rank}: class={CLASS_NAMES[class_id]}, confidence={float(score):.4f}, "
            f"bbox_xyxy={[float(x1), float(y1), float(x2), float(y2)]}"
        )
else:
    print("No detections exceeded the confidence threshold in the sample inference.")

preview_written = cv2.imwrite(str(PREVIEW_PATH), annotated_bgr)
assert preview_written, f"Failed to write preview image to {PREVIEW_PATH}"
assert PREVIEW_PATH.exists(), f"Expected preview image was not found: {PREVIEW_PATH}"
print(f"Saved preview image to {PREVIEW_PATH}")
print("Section 7 completed: the ONNX Runtime sanity check and preview generation finished.")

ONNX Runtime version for sanity check: 1.26.0
Available ONNX Runtime providers: ['AzureExecutionProvider', 'CPUExecutionProvider']
Session providers: ['CPUExecutionProvider']
Input tensor details:
name=images, shape=[1, 3, 640, 640], type=tensor(float)
Output tensor details:
name=output0, shape=[1, 300, 6], type=tensor(float)
Sample test image: 0017.jpg
Letterbox resize scale: 0.666667, padding: (0, 120)
Prepared ONNX tensor shape: (1, 3, 640, 640)
Raw output tensor shape: (1, 300, 6)
Expected native YOLO26 end-to-end layout for detection is approximately (1, 300, 6).
Final detections above confidence 0.25: 1
No manual NMS is applied here because the end-to-end export already produces final detections.
Detection 1: class=dirt_defect, confidence=0.9732, bbox_xyxy=[377.7900695800781, 113.70039367675781, 788.4832763671875, 512.3661499023438]
Saved preview image to C:\Users\mepix\coding\detectx\exported_models\yolo26_onnx_end2end_two_class_retrain_v4_rgb_20260610_150601\preview_20260610_15

## Section 8 - Summarize Exported Artifacts

This section lists every file stored under the ONNX export directory, prints the saved metrics summaries for both the PyTorch and ONNX evaluations, and shows the current dataset manifest so the full 2-class workflow state is easy to inspect.

In [10]:
import csv
import json

TRAINING_SUMMARY_PATH = EXPORT_DIR / "training_summary.json"
RESULTS_CSV_PATH = RUNS_DIR / RUN_NAME / "results.csv"


artifact_paths = sorted(path for path in EXPORT_DIR.rglob("*") if path.is_file())
assert artifact_paths, f"No artifacts were found in {EXPORT_DIR}"
for artifact_path in artifact_paths:
    artifact_size_mb = artifact_path.stat().st_size / (1024 ** 2)
    print(f"{artifact_path.relative_to(EXPORT_DIR)} -> {artifact_size_mb:.2f} MB")

print("PyTorch test metrics:")
print(json.dumps(PT_TEST_METRICS, indent=2))
print("ONNX test metrics:")
print(json.dumps(ONNX_TEST_METRICS, indent=2))


def _maybe_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return value


def summarize_results_csv(results_csv_path):
    if not results_csv_path.exists():
        return None
    with results_csv_path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    if not rows:
        return {
            "path": str(results_csv_path),
            "row_count": 0,
            "columns": [],
            "last_row": None,
            "best_metric_name": None,
            "best_row": None,
        }

    normalized_rows = [
        {key: _maybe_float(value) for key, value in row.items()}
        for row in rows
    ]
    best_metric_name = None
    best_row = None
    for candidate_key in ("metrics/mAP50-95(B)", "metrics/mAP50(B)", "fitness"):
        if candidate_key not in normalized_rows[0]:
            continue

        def _score(row):
            value = row.get(candidate_key)
            return value if isinstance(value, (int, float)) else float("-inf")

        best_metric_name = candidate_key
        best_row = max(normalized_rows, key=_score)
        break

    return {
        "path": str(results_csv_path),
        "row_count": len(normalized_rows),
        "columns": list(rows[0].keys()),
        "last_row": normalized_rows[-1],
        "best_metric_name": best_metric_name,
        "best_row": best_row,
    }


dataset_manifest_snapshot = json.loads(DATASET_MANIFEST_PATH.read_text(encoding="utf-8"))
training_summary = {
    "run_name": RUN_NAME,
    "device": DEVICE,
    "source_dataset_dir": str(SOURCE_DATASET_DIR),
    "filtered_dataset_dir": str(FILTERED_DATASET_DIR),
    "structured_dataset_dir": str(STRUCTURED_DATASET_DIR),
    "data_yaml_path": str(DATA_YAML_PATH),
    "dataset_manifest_path": str(DATASET_MANIFEST_PATH),
    "best_pt_path": str(BEST_PT_PATH),
    "onnx_path": str(ONNX_PATH),
    "preview_path": str(PREVIEW_PATH),
    "pt_test_metrics": PT_TEST_METRICS,
    "onnx_test_metrics": ONNX_TEST_METRICS,
    "artifact_files": [
        {
            "path": str(path.relative_to(EXPORT_DIR)),
            "size_mb": round(path.stat().st_size / (1024 ** 2), 4),
        }
        for path in artifact_paths
    ],
    "dataset_manifest": dataset_manifest_snapshot,
    "results_csv_summary": summarize_results_csv(RESULTS_CSV_PATH),
}
TRAINING_SUMMARY_PATH.write_text(json.dumps(training_summary, indent=2), encoding="utf-8")

print("Dataset manifest snapshot:")
print(json.dumps(dataset_manifest_snapshot, indent=2))
print(f"Training summary written to: {TRAINING_SUMMARY_PATH}")
print(f"ONNX model ready for deployment: {ONNX_PATH}")
print(f"Preview image ready for inspection: {PREVIEW_PATH}")
print("Section 8 completed: exported artifacts and metrics were summarized.")


preview_20260610_150601.jpg -> 0.09 MB
yolo26n_defects_2class_end2end_retrain_v4_rgb_20260610_150601.onnx -> 9.35 MB
yolo26n_defects_2class_end2end_retrain_v4_rgb_20260610_150601_best.pt -> 5.18 MB
PyTorch test metrics:
{
  "mAP50": 0.9898300062441421,
  "mAP50-95": 0.9665642219879553,
  "precision": 0.9760117732647651,
  "recall": 0.9647059755774001
}
ONNX test metrics:
{
  "mAP50": 0.9888169621633247,
  "mAP50-95": 0.9599463738228222,
  "precision": 0.9747905451223964,
  "recall": 0.9708495393839334
}
Dataset manifest snapshot:
{
  "config_signature": {
    "source_dataset_dir": "C:\\Users\\mepix\\coding\\detectx\\raw_dataset_balanced_retrain_v4_rgb",
    "filtered_dataset_dir": "C:\\Users\\mepix\\coding\\detectx\\raw_dataset_balanced_retrain_v4_rgb_two_class",
    "structured_dataset_dir": "C:\\Users\\mepix\\coding\\detectx\\structured_dataset_two_class_retrain_v4_rgb",
    "kept_class_ids": [
      0,
      1
    ],
    "dropped_class_ids": [
      2
    ],
    "class_names": {
   

## Section 9 - Deploying on Raspberry Pi

Use the following command on the Raspberry Pi to install the runtime dependencies:

```bash
python3 -m pip install onnxruntime opencv-python numpy
```

Apply the same preprocessing used in this notebook before inference: letterbox-resize the source image while preserving aspect ratio, pad to `640 x 640` with a constant value of `114`, convert from BGR to RGB, normalize to the range `0.0` to `1.0`, transpose from `HWC` to `CHW`, and add a batch dimension so the final input tensor shape is `(1, 3, 640, 640)`.

Check the ONNX model input tensor details on the Pi. Feed a `float32` tensor with shape `(1, 3, 640, 640)` that matches the preprocessing shown in the sanity-check cell.

The native YOLO26 end-to-end detection output is expected to be approximately `(1, 300, 6)`, where each row is a final detection in the form `[x1, y1, x2, y2, confidence, class_id]` in `xyxy` format.

Do not apply manual NMS after inference. Native YOLO26 end-to-end export already produces final detections directly, so deployment only needs a confidence threshold.

The class order is:

1. `undefected`
2. `dirt_defect`
